# 02 - LLM Knowledge Extraction

**Task 1, Fase 2:** Mengekstrak entitas dan relasi dari chunks UU ITE menggunakan Gemini API, lalu memvalidasi dan mendeduplikasi hasilnya.

**Pipeline:** `Chunks → LLM Extract → Validate → Deduplicate`

In [11]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
print(f'Project root: {PROJECT_ROOT}')
print(f'API Key: {GEMINI_API_KEY[:10]}...' if GEMINI_API_KEY else 'ERROR: No API key found!')

Project root: d:\TA\llm-driven-legal-kg-visualization
API Key: AIzaSyB38G...


## Step 0: Configure PROMPT_ID

Set `PROMPT_ID` to select which prompt to use from the **KG_EXTRACTION_PROMPT** sheet in Google Sheets.

The sheet should have columns: `PROMPT_ID`, `SYSTEM_PROMPT`, `USER_PROMPT`

In [12]:
# === Configure PROMPT_ID ===
# Change this to select a different prompt version from Google Sheets
PROMPT_ID = "PROMPT_3"  # e.g., "PROMPT_1", "PROMPT_2"

print(f'Selected PROMPT_ID: {PROMPT_ID}')

Selected PROMPT_ID: PROMPT_3


In [13]:
# === Fetch Prompt from Google Sheets ===
from modules.prompt_fetcher import fetch_kg_extraction_prompt

prompt_data = fetch_kg_extraction_prompt(PROMPT_ID)

SYSTEM_PROMPT = prompt_data['SYSTEM_PROMPT']
USER_PROMPT_TEMPLATE = prompt_data['USER_PROMPT']

print(f'✅ Loaded prompt: {PROMPT_ID}')
print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'User prompt template length: {len(USER_PROMPT_TEMPLATE)} chars')
print(f'\n--- System Prompt Preview (first 300 chars) ---')
print(SYSTEM_PROMPT[:300])
print(f'\n--- User Prompt Template Preview ---')
print(USER_PROMPT_TEMPLATE[:200])

2026-04-25 08:00:15,992 - INFO - Fetching prompt 'PROMPT_3' from sheet 'KG_EXTRACTION_PROMPT'...
2026-04-25 08:00:15,997 - INFO - Retrieving worksheet 'KG_EXTRACTION_PROMPT' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-04-25 08:00:18,662 - INFO - Successfully loaded 4 rows from worksheet 'KG_EXTRACTION_PROMPT'.
2026-04-25 08:00:18,708 - INFO - Loaded prompt 'PROMPT_3': ['PROMPT_ID', 'SYSTEM_PROMPT', 'USER_PROMPT', 'NOTES']


✅ Loaded prompt: PROMPT_3
System prompt length: 5135 chars
User prompt template length: 148 chars

--- System Prompt Preview (first 300 chars) ---
You are a Knowledge Graph extractor for Indonesian legal documents.
Given a chunk of legal text, extract entities (nodes) and relationships (edges) according to the ontology below.

## Valid Node Types

| Type | Description | Example Labels |
|------|-------------|----------------|
| Regulasi | The 

--- User Prompt Template Preview ---
Extract all entities and relationships from the following Indonesian legal text.
Document ID: {document_id}

<LEGAL_TEXT>
{chunk_text}
</LEGAL_TEXT>


## Step 1: Load Chunks

In [14]:
# Load chunks dari Fase 1
CHUNKS_PATH = 'data/chunks/POJK_11_2022_chunks.json'

with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

chunks = chunks_data['chunks']
print(f'Document: {chunks_data["document_id"]}')
print(f'Total chunks: {len(chunks)}')
print(f'Token stats: {chunks_data.get("stats", {})}')

Document: POJK_11_2022
Total chunks: 24
Token stats: {'min_tokens': 177, 'max_tokens': 2737, 'avg_tokens': 852.9}


## Step 2: Extract KG Triples with Gemini

Mengirim chunks ke Gemini API dengan system prompt ontologi. Output: nodes (entities) dan edges (relations).

⚠️ **Proses ini membutuhkan API calls dan memakan waktu beberapa menit.**

Uses prompt from Google Sheets (PROMPT_ID configured above).

In [15]:
from pipeline.transform.llm_extractor import extract_all_triples

# Extract triples using prompt from Google Sheets
triples_path = extract_all_triples(
    chunks_path=CHUNKS_PATH,
    output_dir='data/triples',
    api_key=GEMINI_API_KEY,
    model_name='gemini-2.5-flash',
    batch_size=3,
    max_retries=3,
    delay_between_calls=2.0,
    system_prompt=SYSTEM_PROMPT,
    user_prompt_template=USER_PROMPT_TEMPLATE,
    prompt_id=PROMPT_ID,
)

# Load results
with open(triples_path, 'r', encoding='utf-8') as f:
    triples_data = json.load(f)

print(f'\n=== Extraction Summary ===')
print(f'Prompt ID: {PROMPT_ID}')
print(f'Total Nodes: {triples_data["total_nodes"]}')
print(f'Total Edges: {triples_data["total_edges"]}')
print(f'Errors: {len(triples_data.get("errors", []))}')
print(f'Saved to: {triples_path}')

Extracting POJK_11_2022:   0%|          | 0/8 [00:00<?, ?it/s]

Extracting POJK_11_2022: 100%|██████████| 8/8 [12:49<00:00, 96.19s/it] 



=== Extraction Summary ===
Prompt ID: PROMPT_3
Total Nodes: 562
Total Edges: 931
Errors: 0
Saved to: data/triples\POJK_11_2022_PROMPT_3_triples.json


In [16]:
# Preview nodes by type
from collections import Counter

node_types = Counter(n['type'] for n in triples_data['nodes'])
edge_types = Counter(e['type'] for e in triples_data['edges'])

print('=== Node Types ===')
for t, count in node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Edge Types ===')
for t, count in edge_types.most_common():
    print(f'  {t:20s}: {count}')

# Sample nodes
print(f'\n=== Sample Nodes (first 10) ===')
for n in triples_data['nodes'][:10]:
    print(f'  [{n["type"]}] {n["label"]}')

=== Node Types ===
  PerbuatanHukum      : 199
  Ayat                : 154
  Pasal               : 81
  EntitasHukum        : 51
  Sanksi              : 30
  Bagian              : 17
  Bab                 : 16
  Regulasi            : 8
  KonsepHukum         : 6

=== Edge Types ===
  BERLAKU_UNTUK       : 216
  MERUJUK             : 208
  MENGATUR            : 200
  MEMILIKI_AYAT       : 154
  MEMUAT              : 106
  MENETAPKAN_SANKSI   : 38
  MENDEFINISIKAN      : 9

=== Sample Nodes (first 10) ===
  [Regulasi] Peraturan Otoritas Jasa Keuangan Nomor 11 Tahun 2022
  [Bab] BAB I KETENTUAN UMUM
  [Pasal] Pasal 1
  [KonsepHukum] Bank Umum
  [KonsepHukum] Teknologi Informasi (TI)
  [KonsepHukum] Sistem Elektronik
  [KonsepHukum] Pusat Data
  [KonsepHukum] Pusat Pemulihan Bencana
  [KonsepHukum] Rencana Pemulihan Bencana
  [EntitasHukum] Direksi


## Step 3: Validate Triples

In [17]:
from pipeline.transform.validator import validate_triples_file

validated_path = validate_triples_file(
    input_path=triples_path,
    output_dir='data/validated',
    strict=False,  # Set True for stricter edge type checking
    prompt_id=PROMPT_ID,
)

with open(validated_path, 'r', encoding='utf-8') as f:
    validated_data = json.load(f)

print(f'=== Validation Summary ===')
print(f'Nodes: {validated_data["total_nodes"]} (removed {validated_data["removed_nodes"]})')
print(f'Edges: {validated_data["total_edges"]} (removed {validated_data["removed_edges"]})')
print(f'Saved to: {validated_path}')

# Show errors if any
error_log = validated_path.replace('_triples.json', '_errors.log')
if os.path.exists(error_log):
    with open(error_log, 'r', encoding='utf-8') as f:
        errors = f.read()
    print(f'\n=== Errors ({errors.count(chr(10))} lines) ===')
    print(errors[:1000])

=== Validation Summary ===
Nodes: 562 (removed 0)
Edges: 917 (removed 14)
Saved to: data/validated\POJK_11_2022_PROMPT_3_triples.json

=== Errors (18 lines) ===
Validation Errors for POJK_11_2022
Total errors: 14

1. Edge target not found: Ayat_36_1 -[MERUJUK]-> Pasal_35_ayat_2
2. Edge target not found: Ayat_36_2 -[MERUJUK]-> Pasal_35_ayat_2
3. Edge target not found: Ayat_36_3 -[MERUJUK]-> Pasal_35_ayat_3
4. Edge target not found: Ayat_36_3 -[MERUJUK]-> Pasal_35_ayat_4
5. Edge target not found: Ayat_40_1 -[MERUJUK]-> Pasal_35_ayat_2
6. Edge target not found: Ayat_40_1 -[MERUJUK]-> Pasal_39_ayat_4
7. Edge target not found: Ayat_41_1 -[MERUJUK]-> Pasal_35_ayat_2
8. Edge target not found: Ayat_41_1 -[MERUJUK]-> Pasal_39_ayat_4
9. Edge target not found: Ayat_42_1 -[MERUJUK]-> Pasal_35_ayat_1
10. Edge target not found: Ayat_42_1 -[MERUJUK]-> Pasal_36_ayat_3
11. Edge target not found: Ayat_42_1 -[MERUJUK]-> Pasal_39_ayat_1
12. Edge target not found: Ayat_42_2 -[MERUJUK]-> Pasal_35_ayat_1
13.

## Step 4: Deduplicate Entities

In [ ]:
from pipeline.transform.deduplicator import deduplicate_triples_file

deduped_path = deduplicate_triples_file(
    input_path=validated_path,
    output_dir='data/deduped',
    similarity_threshold=0.85,
    prompt_id=PROMPT_ID,
)

with open(deduped_path, 'r', encoding='utf-8') as f:
    deduped_data = json.load(f)

print(f'=== Deduplication Summary ===')
print(f'Nodes: {deduped_data["nodes_before"]} → {deduped_data["total_nodes"]} (merged {deduped_data["nodes_merged"]})')
print(f'Edges: {deduped_data["edges_before"]} → {deduped_data["total_edges"]}')
print(f'Saved to: {deduped_path}')

=== Deduplication Summary ===
Nodes: 562 → 511 (merged 17)
Edges: 917 → 915
Saved to: data/deduped\POJK_11_2022_PROMPT_3_triples.json


In [19]:
# Final KG overview
print('=== Final Knowledge Graph ===')
final_node_types = Counter(n['type'] for n in deduped_data['nodes'])
final_edge_types = Counter(e['type'] for e in deduped_data['edges'])

print(f'\nNode Types ({deduped_data["total_nodes"]} total):')
for t, count in final_node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\nEdge Types ({deduped_data["total_edges"]} total):')
for t, count in final_edge_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Sample Triples ===')
node_map = {n['id']: n['label'] for n in deduped_data['nodes']}
for e in deduped_data['edges'][:15]:
    src = node_map.get(e.get('source_id', e.get('source', '')), e.get('source_id', '?'))
    tgt = node_map.get(e.get('target_id', e.get('target', '')), e.get('target_id', '?'))
    print(f'  {src} —[{e["type"]}]→ {tgt}')

=== Final Knowledge Graph ===

Node Types (511 total):
  PerbuatanHukum      : 198
  Ayat                : 154
  Pasal               : 73
  EntitasHukum        : 32
  Bagian              : 16
  Bab                 : 14
  Sanksi              : 12
  Regulasi            : 6
  KonsepHukum         : 6

Edge Types (915 total):
  BERLAKU_UNTUK       : 216
  MENGATUR            : 200
  MERUJUK             : 194
  MEMILIKI_AYAT       : 154
  MEMUAT              : 104
  MENETAPKAN_SANKSI   : 38
  MENDEFINISIKAN      : 9

=== Sample Triples ===
  Peraturan Otoritas Jasa Keuangan Nomor 11/POJK.03/2022 tentang Penyelenggaraan Teknologi Informasi oleh Bank Umum —[MEMUAT]→ BAB I KETENTUAN UMUM
  BAB I KETENTUAN UMUM —[MEMUAT]→ Pasal 1
  Pasal 1 —[MENDEFINISIKAN]→ Bank Umum
  Pasal 1 —[MENDEFINISIKAN]→ Teknologi Informasi (TI)
  Pasal 1 —[MENDEFINISIKAN]→ Sistem Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Pusat Data
  Pasal 1 —[MENDEFINISIKAN]→ Pusat Pemulihan Bencana
  Pasal 1 —[MENDEFINISIKAN]→ Rencana 

## Summary

Fase 2 selesai! Output tersimpan di:
- `data/triples/{doc_id}_{prompt_id}_triples.json` — Raw LLM extraction
- `data/validated/{doc_id}_{prompt_id}_triples.json` — Validated (invalid types removed)
- `data/deduped/{doc_id}_{prompt_id}_triples.json` — Deduplicated (final)

**Prompt used:** Loaded from Google Sheets sheet `KG_EXTRACTION_PROMPT` with PROMPT_ID configured above.

**Next:** Run `03_neo4j_ingestion.ipynb` to load into Neo4j.